# **🏡 Project: The House Price Engine 📈**

## **🌟 What are we building?**

Welcome to the **House Price Prediction Project**! Pricing real estate is notoriously tricky. It’s a mix of hard facts (square footage) and soft variables (neighborhood vibes). For banks, buyers, and platforms like Zillow, getting this wrong by even 5% can mean leaving tens of thousands of dollars on the table.

In this project, we are stepping into the shoes of a **Data Scientist** to build a machine learning model that takes the guesswork out of property values. We will use `pandas` to wrangle messy raw housing data and build a predictive engine that accurately prices homes based on their actual features.

---

## **🗺️ The Roadmap: How we get it done**

We aren't just throwing data at an algorithm and hoping for the best. We’re building a clean, step-by-step pipeline across four key phases:

* **1. Digging into the Data (EDA) 🔍**
  * We'll use `pandas` to pull in our CSV files, check out what data types we're dealing with, and hunt down missing values. 
  * We'll look at the distribution of house prices to see if luxury homes are skewing our numbers, and find out which variables actually correlate with a higher price tag.

* **2. Cleaning & Feature Engineering 🛠️**
  * Raw data is never perfect. We will use `pandas` methods to fill in missing gaps and drop weird outliers (like a massive mansion sold for dirt cheap).
  * We'll create smarter features that the model can understand—like combining individual porch and deck metrics into a single "Total Outdoor Space" variable, or calculating exactly how old a house was the year it was sold.

* **3. Training the Models 🤖**
  * Because we are predicting a continuous number (price), this is a **Regression** problem.
  * We’ll start with a straightforward linear model to set a baseline score. Once that’s locked in, we’ll unleash heavy-hitting gradient-boosted trees like **XGBoost** and **LightGBM** to handle the complex, non-linear relationships in the data.

* **4. Keeping Evaluation Realistic 📊**
  * We will test our models using **RMSLE** (Root Mean Squared Log Error). Why? Because a \$20,000 mistake on a \$100,000 starter home is a disaster, but a \$20,000 mistake on a \$2,000,000 mansion is practically a rounding error. Log error keeps our penalties fair across all price brackets.

---

## **💡 Coding Standards**

We are writing code that looks like it belongs in a production environment, not just a sandbox:

* **Readable & Modular:** No giant blocks of messy code. We’ll write clean, reusable python functions with clear descriptions.
* **Bulletproof Integrity:** We will explicitly validate our data shapes and types using `pandas` before passing anything to our machine learning models. 
* **Scalable Thinking:** The logic we write for this dataset will be clean enough to easily scale up to enterprise-level data down the road.

### 🚀 Automated Data Ingestion

To ensure maximum reproducibility and maintain clean versioning, we pull the dataset directly using Kaggle's tools. This automated process fetches the raw housing feature records—tracking structural properties, location metrics, and sales history—directly into our environment.

* **Dataset Credit:** Vedat Gül via Kaggle (*House Prices Prediction / Advanced Regression Techniques*).
* **Source Notebook/Data:** [Kaggle Notebook Link](https://www.kaggle.com/datasets/fratzcan/usa-house-prices)

### 📥 Loading the Dataset and Libraries

Before we start, we need to install the necessary Python libraries and **load the dataset**.


In [1]:
# Install the required libraries
%pip install kagglehub pandas numpy matplotlib seaborn scikit-learn lightgbm xgboost sweetviz scikit-optimize jupyterlab nbconvert imblearn xgboost joblib -q

# Install and update the watermark package to display environment and library version information
%pip install -q -U watermark

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================================
# 📦 DEPENDENCIES
# ============================================================
import os
import warnings
import logging
import joblib
import numpy as np
import pandas as pd

# ✅ Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import sweetviz as sv

# ✅ Preprocessing
from sklearn.impute          import SimpleImputer
from sklearn.preprocessing   import StandardScaler, OneHotEncoder
from sklearn.compose         import ColumnTransformer
from sklearn.pipeline        import Pipeline

# ✅ Models
from sklearn.dummy           import DummyRegressor
from sklearn.linear_model    import Ridge
from sklearn.ensemble        import RandomForestRegressor, HistGradientBoostingRegressor
import xgboost  as xgb
import lightgbm as lgb

# ✅ Evaluation
from sklearn.metrics         import root_mean_squared_error, mean_squared_log_error, mean_absolute_error, r2_score
from skopt import BayesSearchCV
from skopt.space import Integer, Real, Categorical

# ✅ Validation
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV

# ✅ Warning filters
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', message=".*findfont.*")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
plt.style.use('dark_background')

c:\Users\LarTI\OneDrive\Desktop\Projects\House_Prices_Prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load the watermark extension to log the environment state
%reload_ext watermark

# Display professional metadata tracking our data engineering stack
%watermark -a "Maykon - 🏡 The House Price Engine" -d -u -v -p pandas,numpy,matplotlib,seaborn,scikit-learn,lightgbm,xgboost,sweetviz,scikit-optimize,jupyterlab,nbconvert,imblearn,xgboost,joblib

Author: Maykon - 🏡 The House Price Engine

Last updated: 2026-07-23

Python implementation: CPython
Python version       : 3.13.7
IPython version      : 9.14.1

pandas         : 2.3.3
numpy          : 2.3.5
matplotlib     : 3.10.0
seaborn        : 0.13.2
scikit-learn   : 1.9.0
lightgbm       : 4.6.0
xgboost        : 3.3.0
sweetviz       : 2.3.3
scikit-optimize: 0.10.2
jupyterlab     : 4.6.2
nbconvert      : 7.17.1
imblearn       : 0.0
joblib         : 1.5.3



In [4]:
# Download latest version
path = kagglehub.dataset_download("fratzcan/usa-house-prices")

print("📦 Path to dataset files:", path)

📦 Path to dataset files: C:\Users\LarTI\.cache\kagglehub\datasets\fratzcan\usa-house-prices\versions\1


In [5]:
# --- LOCATING AND READING THE CSV ---
# List out all files inside the downloaded repository path to spot the target file
all_files = os.listdir(path)
print("📂 Files discovered in directory:", all_files)

# Filter out all CSV files dynamically
csv_files = [file for file in all_files if file.endswith('.csv')]

if len(csv_files) == 0:
    raise FileNotFoundError("❌ Critical Error: No CSV files found in the downloaded folder!")
else:
    # Grab the primary CSV file found
    csv_filename = csv_files[0]
    full_csv_path = os.path.join(path, csv_filename)
    print(f"🎯 Target CSV located: {csv_filename}")

📂 Files discovered in directory: ['USA Housing Dataset.csv']
🎯 Target CSV located: USA Housing Dataset.csv


In [6]:
# Ingest the dataset into a pandas DataFrame
df = pd.read_csv(full_csv_path)
print(f"✅ Dataset successfully loaded! Shape: {df.shape[0]} rows, {df.shape[1]} columns.")

✅ Dataset successfully loaded! Shape: 4140 rows, 18 columns.


In [7]:
# Display the first 5 records to see our column properties and labels
df.head()

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,2014-05-09 00:00:00,376000.0,3.0,2.00,1340,1384,3.0,0,0,3,1340,0,2008,0,9245-9249 Fremont Ave N,Seattle,WA 98103,USA
1,2014-05-09 00:00:00,800000.0,4.0,3.25,3540,159430,2.0,0,0,3,3540,0,2007,0,33001 NE 24th St,Carnation,WA 98014,USA
2,2014-05-09 00:00:00,2238888.0,5.0,6.50,7270,130017,2.0,0,0,3,6420,850,2010,0,7070 270th Pl SE,Issaquah,WA 98029,USA
3,2014-05-09 00:00:00,324000.0,3.0,2.25,998,904,2.0,0,0,3,798,200,2007,0,820 NW 95th St,Seattle,WA 98117,USA
4,2014-05-10 00:00:00,549900.0,5.0,2.75,3060,7015,1.0,0,0,5,1600,1460,1979,0,10834 31st Ave SW,Seattle,WA 98146,USA


In [8]:
df.tail() #Displays the last 5 rows of the DataFrame df.

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
4135,2014-07-09 00:00:00,308166.666667,3.0,1.75,1510,6360,1.0,0,0,4,1510,0,1954,1979,501 N 143rd St,Seattle,WA 98133,USA
4136,2014-07-09 00:00:00,534333.333333,3.0,2.50,1460,7573,2.0,0,0,3,1460,0,1983,2009,14855 SE 10th Pl,Bellevue,WA 98007,USA
4137,2014-07-09 00:00:00,416904.166667,3.0,2.50,3010,7014,2.0,0,0,3,3010,0,2009,0,759 Ilwaco Pl NE,Renton,WA 98059,USA
4138,2014-07-10 00:00:00,203400.000000,4.0,2.00,2090,6630,1.0,0,0,3,1070,1020,1974,0,5148 S Creston St,Seattle,WA 98178,USA
4139,2014-07-10 00:00:00,220600.000000,3.0,2.50,1490,8102,2.0,0,0,4,1490,0,1990,0,18717 SE 258th St,Covington,WA 98042,USA


In [9]:
df.describe(include='all').T  # Generates descriptive statistics for all numeric columns in your DataFrame.

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
date,4140,68,2014-06-23 00:00:00,142,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price,4140.0,NaN,NaN,NaN,553062.877289,583686.452245,0.0,320000.0,460000.0,659125.0,26590000.0
bedrooms,4140.0,NaN,NaN,NaN,3.400483,0.903939,0.0,3.0,3.0,4.0,8.0
bathrooms,4140.0,NaN,NaN,NaN,2.163043,0.784733,0.0,1.75,2.25,2.5,6.75
sqft_living,4140.0,NaN,NaN,NaN,2143.638889,957.481621,370.0,1470.0,1980.0,2620.0,10040.0
sqft_lot,4140.0,NaN,NaN,NaN,14697.638164,35876.838123,638.0,5000.0,7676.0,11000.0,1074218.0
floors,4140.0,NaN,NaN,NaN,1.51413,0.534941,1.0,1.0,1.5,2.0,3.5
waterfront,4140.0,NaN,NaN,NaN,0.007488,0.086219,0.0,0.0,0.0,0.0,1.0
view,4140.0,NaN,NaN,NaN,0.246618,0.790619,0.0,0.0,0.0,0.0,4.0
condition,4140.0,NaN,NaN,NaN,3.452415,0.678533,1.0,3.0,3.0,4.0,5.0


In [10]:
# Information about the dataframe
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4140 entries, 0 to 4139
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           4140 non-null   object 
 1   price          4140 non-null   float64
 2   bedrooms       4140 non-null   float64
 3   bathrooms      4140 non-null   float64
 4   sqft_living    4140 non-null   int64  
 5   sqft_lot       4140 non-null   int64  
 6   floors         4140 non-null   float64
 7   waterfront     4140 non-null   int64  
 8   view           4140 non-null   int64  
 9   condition      4140 non-null   int64  
 10  sqft_above     4140 non-null   int64  
 11  sqft_basement  4140 non-null   int64  
 12  yr_built       4140 non-null   int64  
 13  yr_renovated   4140 non-null   int64  
 14  street         4140 non-null   object 
 15  city           4140 non-null   object 
 16  statezip       4140 non-null   object 
 17  country        4140 non-null   object 
dtypes: float

In [11]:
df.shape # (rows, columns)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 4140
Number of columns: 18


In [12]:
df.dtypes #Displays the data type of each column in the DataFrame.

date              object
price            float64
bedrooms         float64
bathrooms        float64
sqft_living        int64
sqft_lot           int64
floors           float64
waterfront         int64
view               int64
condition          int64
sqft_above         int64
sqft_basement      int64
yr_built           int64
yr_renovated       int64
street            object
city              object
statezip          object
country           object
dtype: object

In [13]:
df.columns #Returns a list (Index object) containing the names of all columns in the DataFrame.

Index(['date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
       'floors', 'waterfront', 'view', 'condition', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'street', 'city',
       'statezip', 'country'],
      dtype='object')

### 🧹 Structural cleaning

After loading the dataset and reviewing its structure with 'df.info()' and, the next step is to identify missing values in the dataset.  

We use:

In [14]:
df.isna().sum() # Count missing values per column

date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
street           0
city             0
statezip         0
country          0
dtype: int64

In [15]:
df.drop_duplicates(inplace=True) # Remove duplicate rows from the DataFrame df.
print("Number of duplicate rows Now:", df.duplicated().sum()) # After dropping duplicates, check again to confirm that there are no duplicate rows remaining in the DataFrame df.

Number of duplicate rows Now: 0


In [16]:
# ============================================================
# 🧹 COLUMN NAME SANITIZATION
# ============================================================
df.columns = (
    df.columns
    .str.lower()                                        # lowercase everything
    .str.strip()                                        # remove leading/trailing whitespace
    .str.replace(r"[^a-z0-9]+", "_", regex=True)       # replace anything not a letter/number with _
    .str.strip("_")                                     # remove leading/trailing underscores
)

print("✅ Columns sanitized:")
print(df.columns.tolist())

✅ Columns sanitized:
['date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'street', 'city', 'statezip', 'country']


#### 🧠 Create the profiling report

In [17]:
# 1. Generate the Sweetviz report
report = sv.analyze(df, target_feat='price')

# 2. Save the report to an HTML file
report.show_html('House_Prices_Report.html')

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)


Report House_Prices_Report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


#### 🎯 Defining Features (X) and Target (y)

In supervised machine learning, every dataset is divided into two roles:

- **Features (`X`)** → the input variables the model uses to learn patterns (square footage, location, number of bedrooms).
- **Target (`y`)** → the outcome we want to predict — in our case, **house price**.

> ⚠️ **We split the data here, before outlier removal.** This is a hard rule. Doing any transformation on the full dataset before splitting causes **data leakage** — the model indirectly sees test set information during training, producing results that look great in the notebook but fail in the real world.

In [18]:
# Here we are separating the features (X) from the target variable (y). 
# The target variable is 'price', which indicates the price of the house.
# The features (X) are all the other columns in the DataFrame dfc, which will be used to predict the target variable.
X = df.drop(columns=['price'])  # X = all columns except price
y = df['price']                  # y = ONLY the price column

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### 🧹 Outlier Filtering using Interquartile Range (IQR)

To prevent extreme house prices from distorting model training, we filter out statistical outliers from `y_train` using the **1.5 × IQR rule**:

* **1. Measure Data Spread:**
  * `Q1` (25th percentile) and `Q3` (75th percentile) define the middle 50% range of house prices.
  * `IQR = Q3 - Q1` measures the middle spread of prices.

* **2. Define Acceptable Boundaries:**
  * **Lower Bound:** $Q1 - (1.5 \times IQR)$
  * **Upper Bound:** $Q3 + (1.5 \times IQR)$

* **3. Apply Mask & Filter:**
  * Keep only the houses whose prices fall strictly between the lower and upper thresholds.
  * Apply the resulting boolean mask to **both** `X_train` and `y_train` to maintain matching row indices.

> 🔒 **Data Leakage Safeguard:** Outliers are filtered **strictly on `X_train` / `y_train`** after the train-test split. The test set (`X_test` / `y_test`) is untouched so model evaluation remains unbiased.

In [19]:
# After the split — remove outliers on training data only
before = X_train.shape[0]

# Step 1: Remove impossible prices — a house cannot cost $0
valid_price_mask = y_train > 0
X_train = X_train[valid_price_mask]
y_train = y_train[valid_price_mask]

# Step 2: Remove extreme price outliers using IQR
Q1 = y_train.quantile(0.25)
Q3 = y_train.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
mask = (y_train >= lower_bound) & (y_train <= upper_bound)
X_train = X_train[mask]
y_train  = y_train[mask]

print(f"Removed {before - X_train.shape[0]} rows (invalid prices + outliers) from training set")
print(f"Price range: ${y_train.min():,.0f} — ${y_train.max():,.0f}")

Removed 217 rows (invalid prices + outliers) from training set
Price range: $7,800 — $1,160,000


#### ❓ Missing Value Analysis

To identify incomplete data across the dataset, we calculate both the absolute count and relative percentage of missing (`NaN`) values for each column:

* **1. Measure Missingness:**
  * `df.isna().sum()` counts the total number of missing (`NaN`) entries per feature.
  * `(df.isna().sum() / len(df)) * 100` converts those counts into percentages relative to the total row count.

* **2. Structure & Filter Summary:**
  * Both metrics are combined into a clean pandas `DataFrame` for comparison.
  * `.query('`Missing Count` > 0')` filters out complete columns, displaying only features that actually contain missing data.
  * `.sort_values(by='Missing Count', ascending=False)` orders the results so features with the highest missingness appear first.

> 💡 **Preprocessing Step:** This analysis guides our imputation strategy inside the column transformer pipeline (e.g., deciding whether to impute numeric features using the `median` or categorical features using the `most_frequent` value).

In [20]:
missing_series = X_train.isna().sum()[X_train.isna().sum() > 0]

if not missing_series.empty:
    plt.figure(figsize=(8, 4))
    sns.barplot(x=missing_series.values, y=missing_series.index, palette='Reds_r')
    plt.title('Missing Value Count per Feature')
    plt.xlabel('Count')
    plt.show()
else:
    print("🎉 No missing values found in the dataset!")

🎉 No missing values found in the dataset!


#### 🛠️ Feature Engineering

This is where you create new, more useful columns from what you already have. Based on your dataset's columns, here's what makes sense:

In [21]:
# Apply feature engineering to both train and test sets separately
# to avoid data leakage — no full dataset transformations after split

for dataset in [X_train, X_test]:
    # Extract sale year from date
    dataset['date']             = pd.to_datetime(dataset['date'])
    dataset['sale_year']        = dataset['date'].dt.year

    # How old was the house when it was sold?
    dataset['house_age']        = dataset['sale_year'] - dataset['yr_built']

    # Was the house ever renovated?
    dataset['was_renovated']    = (dataset['yr_renovated'] > 0).astype(int)

    # How many years since the last update (renovation or original build)?
    dataset['years_since_update'] = dataset.apply(
        lambda row: row['sale_year'] - row['yr_renovated']
        if row['yr_renovated'] > 0
        else row['sale_year'] - row['yr_built'],
        axis=1
    )

# Drop redundant columns — raw values now replaced by engineered features
cols_to_drop = ['country', 'street', 'date', 'yr_built', 'yr_renovated', 'sale_year']
X_train = X_train.drop(columns=cols_to_drop)
X_test  = X_test.drop(columns=cols_to_drop)

# Confirm final shape
print(f"✅ Feature engineering complete!")
print(f"X_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")
print(f"Final columns : {X_train.columns.tolist()}")

✅ Feature engineering complete!
X_train shape : (3095, 15)
X_test shape  : (828, 15)
Final columns : ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 'city', 'statezip', 'house_age', 'was_renovated', 'years_since_update']


#### 🔍 Preprocessing Feature Alignment Check

Before building the `ColumnTransformer` and training pipelines, we perform a sanity check to verify that all specified feature names exist inside `X_train`:

* **1. List Comprehensions & Set Membership:**
  * `[col for col in numeric_features if col in X_train.columns]` filters `numeric_features` to confirm which columns are present in the training set.
  * `[col for col in numeric_features if col not in X_train.columns]` isolates any numerical features that were accidentally dropped, misspelled, or missing.

* **2. Categorical Column Verification:**
  * The same lookup logic is applied to `categorical_features` to ensure all expected text or group columns exist before sending them to `OneHotEncoder`.

* **3. Diagnostic Visual Feedback:**
  * Clear emoji headers (`✅` and `❌`) make it easy to instantly spot pipeline configuration errors or missing columns in the execution output.

> 💡 **Why This Matters:** Running this check prevents silent pipeline failures, key errors, or unexpected column mismatches during target encoding and scaling steps.

In [22]:
# Separate columns by type
numeric_features = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement',
                    'house_age', 'was_renovated', 'years_since_update']

categorical_features = ['city', 'statezip']

In [23]:
print("✅ Numeric columns check:")
print([col for col in numeric_features if col in X_train.columns])

print("\n❌ Missing numeric columns:")
print([col for col in numeric_features if col not in X_train.columns])

print("\n✅ Categorical columns check:")
print([col for col in categorical_features if col in X_train.columns])

print("\n❌ Missing categorical columns:")
print([col for col in categorical_features if col not in X_train.columns])

✅ Numeric columns check:
['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 'house_age', 'was_renovated', 'years_since_update']

❌ Missing numeric columns:
[]

✅ Categorical columns check:
['city', 'statezip']

❌ Missing categorical columns:
[]


#### 🔧 Pipeline
A pipeline is a sequence of automated steps that processes your data and trains your model in the correct order.
**Instead of manually encoding your data, splitting, and then training — the pipeline chains everything together into a single object. You call fit() once and it handles the rest.**
The biggest advantage is safety — it guarantees that encoding is learned only from training data and applied correctly to test data, eliminating data leakage without you having to think about it.
**In short:**
* Without pipeline → you manage every step manually → easy to make mistakes
* With pipeline    → one object manages everything  → safe and reproducible

In [24]:
# --- Numeric pipeline: impute missing values, then scale ---
numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])

# --- Categorical pipeline: impute missing values, then one-hot encode ---
categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])

# Update your ColumnTransformer definition
preprocessor = ColumnTransformer(
    transformers=[('num', numeric_transformer, numeric_features), ('cat', categorical_transformer, categorical_features)], remainder='drop')

#### 🔁 Cross-Validation and Training the Models

To obtain a more reliable estimate of our model’s performance and reduce the dependency on a single data split, we use cross-validation. This technique repeatedly partitions the dataset into multiple training and validation subsets, allowing the model to be trained and evaluated across different data segments.

* The dataset is divided into $K$ folds (e.g., 5 folds).
* In each iteration:
  * $K - 1$ folds are used for training
  * 1 fold is used for validation

The process is repeated $K$ times, and the final performance is computed as the average metric across all folds.

---

We use **`cross_val_score`** paired with a **`KFold`** cross-validation strategy, setting `shuffle=True` alongside a fixed `random_state` to ensure continuous target values are randomly distributed across all folds for robust evaluation and exact reproducibility.

In [25]:
# Correct setup for Regression tasks (House Prices)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

#### 🏁 Baseline
The baseline represents the minimum threshold your model must surpass.
Before building complex algorithms, we ask: "What score would we get if we made the simplest possible guess without looking at any features?" That is our benchmark.

While a classifier predicts the most frequent category, a DummyRegressor predicts the average target value (e.g., the mean house price) for every single prediction. If our machine learning models cannot beat this simple average benchmark, they aren't learning meaningful patterns from the data.

In [26]:
# Baseline pipeline
dummy_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DummyRegressor(strategy='mean'))
])

# CV evaluation
dummy_mae_scores  = -cross_val_score(dummy_pipeline, X_train, y_train, cv=cv, scoring='neg_mean_absolute_error')
dummy_rmse_scores = -cross_val_score(dummy_pipeline, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error')

print(f"📊 Baseline (DummyRegressor) | MAE: ${dummy_mae_scores.mean():,.0f} (+/- ${dummy_mae_scores.std():,.0f}) | RMSE: ${dummy_rmse_scores.mean():,.0f} (+/- ${dummy_rmse_scores.std():,.0f})")

📊 Baseline (DummyRegressor) | MAE: $175,398 (+/- $7,200) | RMSE: $217,153 (+/- $7,408)


**What these numbers mean**

MAE: $175,398
On average, if your model just predicts the mean price for every house, it's off by $178,215 per house. That's the minimum bar your real model must beat.

RMSE: $217,153
Similar story but RMSE penalizes large errors more heavily. A $222,155 average error means some houses are being missed by much more than $178,215.

+/- $5,166 and +/- $5,387
The standard deviation across the 5 folds is small — meaning the baseline is stable and consistent. The dataset is well distributed across folds.

#### 🤖 Model Comparison

We now train and evaluate 3 models using the same cross-validation strategy as the baseline — ensuring a fair, apples-to-apples comparison.

Every model is wrapped in a full pipeline **(preprocessor + model)** so preprocessing is learned only from training folds — no leakage.

The winner is selected automatically based on lowest RMSE.

#### 🛡️ Ridge Regression

Ridge Regression is a linear model built for supervised regression tasks, which means it predicts a continuous target variable — here, the house price.

Unlike a simple linear regression, Ridge adds a penalty term to the loss function to shrink the coefficients of the model. This helps reduce overfitting and improves generalization when features are correlated or when the dataset contains many predictors.

Why Ridge works well here:

* It is fast to train and easy to interpret.
* It handles multicollinearity much better than ordinary least squares.
* It tends to perform well on tabular data when the relationship is mostly linear.
* It provides a strong benchmark before moving to more complex tree-based models.

The model equation can be summarized as:

- Prediction = linear combination of input features
- Regularization penalty = alpha × sum of squared coefficients

In practice:

* `alpha` controls the strength of the regularization.
* Smaller `alpha` → closer to standard linear regression
* Larger `alpha` → stronger coefficient shrinkage, which can reduce variance

In this notebook, Ridge was the best-performing baseline model, so we now tune its `alpha` value using cross-validation to find the best balance between bias and variance.

In [27]:
# Build a Ridge regression model using the existing preprocessing pipeline
ridge_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', Ridge(alpha=1.0))
    ]
)

# Train the model
ridge_pipeline.fit(X_train, y_train)

# Predict on the untouched test set
ridge_pred = ridge_pipeline.predict(X_test)

# Evaluate the model
ridge_mae = mean_absolute_error(y_test, ridge_pred)
ridge_rmse = root_mean_squared_error(y_test, ridge_pred)
ridge_r2 = r2_score(y_test, ridge_pred)

print(f"Ridge MAE : ${ridge_mae:,.0f}")
print(f"Ridge RMSE: ${ridge_rmse:,.0f}")
print(f"Ridge R²  : {ridge_r2:.4f}")

Ridge MAE : $105,859
Ridge RMSE: $214,959
Ridge R²  : 0.5592


#### 🌲 Random Forest

Random Forest builds hundreds of decision trees and combines their votes to make a final prediction. This makes it much better at capturing complex patterns in the data.

In [28]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

rf_mae  = -cross_val_score(rf_pipeline, X_train, y_train, cv=cv, scoring='neg_mean_absolute_error')
rf_rmse = -cross_val_score(rf_pipeline, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error')

print(f"📊 Random Forest | MAE: ${rf_mae.mean():,.0f} (+/- ${rf_mae.std():,.0f}) | RMSE: ${rf_rmse.mean():,.0f} (+/- ${rf_rmse.std():,.0f})")

📊 Random Forest | MAE: $81,484 (+/- $3,002) | RMSE: $116,714 (+/- $6,682)


#### ⚡ XGBoost

XGBoost (Extreme Gradient Boosting) is an advanced implementation of the Gradient Boosting algorithm.

While Random Forest builds many decision trees independently and combines their predictions (bagging), XGBoost builds trees sequentially (boosting). Each new tree is trained to correct the mistakes made by the previous trees, gradually improving the model.

Why we love it: XGBoost is one of the most powerful algorithms for structured (tabular) data and has won many machine learning competitions. It excels at capturing subtle, complex relationships in data while including built-in regularization to help reduce overfitting.

The catch: Because trees are built one after another, training is generally slower than Random Forest. XGBoost also has more hyperparameters to tune, and without proper tuning it can still overfit.

In [29]:
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor(random_state=42, verbosity=0))
])

xgb_mae  = -cross_val_score(xgb_pipeline, X_train, y_train, cv=cv, scoring='neg_mean_absolute_error')
xgb_rmse = -cross_val_score(xgb_pipeline, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error')

print(f"📊 XGBoost | MAE: ${xgb_mae.mean():,.0f} (+/- ${xgb_mae.std():,.0f}) | RMSE: ${xgb_rmse.mean():,.0f} (+/- ${xgb_rmse.std():,.0f})")

📊 XGBoost | MAE: $74,609 (+/- $2,170) | RMSE: $109,633 (+/- $6,544)


#### 📊 LightGBM

LightGBM (Light Gradient Boosting Machine) is a highly optimized implementation of the Gradient Boosting algorithm, designed for speed and efficiency. Like XGBoost, it builds decision trees sequentially, with each new tree learning to correct the mistakes made by the previous ones.

In [30]:
lgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', lgb.LGBMRegressor(random_state=42, verbose=-1))
])

lgb_mae  = -cross_val_score(lgb_pipeline, X_train, y_train, cv=cv, scoring='neg_mean_absolute_error')
lgb_rmse = -cross_val_score(lgb_pipeline, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error')

print(f"📊 LightGBM | MAE: ${lgb_mae.mean():,.0f} (+/- ${lgb_mae.std():,.0f}) | RMSE: ${lgb_rmse.mean():,.0f} (+/- ${lgb_rmse.std():,.0f})")

📊 LightGBM | MAE: $75,581 (+/- $3,260) | RMSE: $108,954 (+/- $7,993)


##### ============================================================
##### 🏆 MODEL COMPARISON
##### ============================================================

In [31]:
results = {
    'Baseline': (dummy_mae_scores.mean(), dummy_rmse_scores.mean()),
    'Ridge': (-cross_val_score(ridge_pipeline, X_train, y_train, cv=cv, scoring='neg_mean_absolute_error' ).mean(),
              -cross_val_score(ridge_pipeline, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error').mean()),
    'Random Forest': (rf_mae.mean(), rf_rmse.mean()),
    'XGBoost': (xgb_mae.mean(), xgb_rmse.mean() ),
    'LightGBM': (lgb_mae.mean(), lgb_rmse.mean()),
}

best_model = min(results, key=lambda name: results[name][0])  # lowest MAE wins

print("=" * 65)
print(f"{'Model':<20} | {'MAE':>12} | {'RMSE':>12}")
print("=" * 65)
for name, (mae, rmse) in results.items():
    trophy = " 🏆" if name == best_model else ""
    print(f"{name:<20} | ${mae:>10,.0f} | ${rmse:>10,.0f}{trophy}")
print("=" * 65)

Model                |          MAE |         RMSE
Baseline             | $   175,398 | $   217,153
Ridge                | $    70,696 | $   101,941 🏆
Random Forest        | $    81,484 | $   116,714
XGBoost              | $    74,609 | $   109,633
LightGBM             | $    75,581 | $   108,954


### 🔧 Hyperparameter Tuning — Ridge (Winner)

Ridge won the model comparison with the lowest MAE and RMSE.
We now tune its key parameter — `alpha` (regularization strength) —
using `GridSearchCV` with the same 5-fold CV strategy to find the
optimal penalty that minimizes RMSE without overfitting.

In [32]:
# Define the parameter grid
param_grid = {
    'model__alpha': [0.01, 0.1, 1, 10, 50, 100, 500, 1000]
}

# Full pipeline with Ridge
ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge())
])

# GridSearchCV with same CV strategy
grid_search = GridSearchCV(
    estimator  = ridge_pipeline,
    param_grid = param_grid,
    cv         = cv,
    scoring    = 'neg_root_mean_squared_error',
    n_jobs     = -1,
    verbose    = 1
)

grid_search.fit(X_train, y_train)

print(f"✅ Best alpha : {grid_search.best_params_['model__alpha']}")
print(f"✅ Best RMSE  : ${-grid_search.best_score_:,.0f}")

Fitting 5 folds for each of 8 candidates, totalling 40 fits
✅ Best alpha : 0.1
✅ Best RMSE  : $101,865


#### 📊 Model Evaluation

Now that tuning is complete, we evaluate the best model on the **test set for the first time**. This is the true measure of how well our model generalizes to completely unseen data.

> 🔒 The test set was never touched during training or tuning.
> This is our one honest look at real-world performance.

In [33]:
best_model = grid_search.best_estimator_

print(f"✅ Best model extracted and ready for final evaluation")
print(f"   Alpha : {grid_search.best_params_['model__alpha']}")
print(f"   RMSE  : ${-grid_search.best_score_:,.0f}")

✅ Best model extracted and ready for final evaluation
   Alpha : 0.1
   RMSE  : $101,865


In [34]:
# ============================================================
# 🔮 TEST SET PREDICTIONS
# ============================================================

y_pred  = best_model.predict(X_test)
y_proba = np.clip(y_pred, 0, None)  # clip negative predictions to 0

print(f"✅ Predictions generated on {X_test.shape[0]} unseen houses")
print(f"   Predicted price range: ${y_pred.min():,.0f} — ${y_pred.max():,.0f}")
print(f"   Actual price range   : ${y_test.min():,.0f} — ${y_test.max():,.0f}")

✅ Predictions generated on 828 unseen houses
   Predicted price range: $60,582 — $1,487,531
   Actual price range   : $0 — $3,000,000


In [35]:
# ============================================================
# 📋 CORE METRICS
# ============================================================
# --- Evaluation 1: Full test set (real world) ---
mae_full  = mean_absolute_error(y_test, y_pred)
rmse_full = root_mean_squared_error(y_test, y_pred)
r2_full   = r2_score(y_test, y_pred)

# --- Evaluation 2: Valid prices only (no $0 houses) ---
mask         = y_test > 0
mae_clean    = mean_absolute_error(y_test[mask], y_pred[mask])
rmse_clean   = root_mean_squared_error(y_test[mask], y_pred[mask])
r2_clean     = r2_score(y_test[mask], y_pred[mask])

print("=" * 55)
print("📋 FINAL TEST METRICS")
print("=" * 55)
print(f"{'Metric':<15} | {'Full Test':>15} | {'Valid Only':>15}")
print("=" * 55)
print(f"{'MAE':<15} | ${mae_full:>13,.0f} | ${mae_clean:>13,.0f}")
print(f"{'RMSE':<15} | ${rmse_full:>13,.0f} | ${rmse_clean:>13,.0f}")
print(f"{'R²':<15} | {r2_full:>15.4f} | {r2_clean:>15.4f}")
print("=" * 55)
print()
print(f"ℹ️  Full test  : {y_test.shape[0]} houses (includes $0 prices)")
print(f"ℹ️  Valid only : {y_test[mask].shape[0]} houses (excludes $0 prices)")

📋 FINAL TEST METRICS
Metric          |       Full Test |      Valid Only
MAE             | $      106,065 | $       99,737
RMSE            | $      217,335 | $      202,934
R²              |          0.5494 |          0.5983

ℹ️  Full test  : 828 houses (includes $0 prices)
ℹ️  Valid only : 818 houses (excludes $0 prices)


_**The honest interpretation**_

**MAE: $99,737 (valid only)**
On average the model is off by ~$100k per house. Given that house prices range up to $3,000,000 that's reasonable — but not great.

**RMSE: $202,934 (valid only)**
RMSE is much higher than MAE — this confirms there are large individual errors on luxury homes above $1.5M that the model can't predict well.

**R²: 0.5983**
The model explains about 60% of price variance. This means 40% of what drives prices isn't captured by your features. Not bad, but room for improvement.

#### **The conclusion to draw**

Ridge is doing a solid job on normal-priced houses but struggling with luxury properties. This is a data problem, not a model problem — you simply don't have enough $1.5M+ houses in training data for the model to learn that range.